# SemEval 2026 Task 13 — End-to-End Kaggle Pipeline

Pipeline hoàn chỉnh: retrain v10 (language-robust gradient ensemble) + ensemble với v5 + CodeBERT, sinh ~13 submission candidates.

**Thời gian dự kiến**: 25-45 phút trên Kaggle CPU (không cần GPU).

## Thiết lập trước khi chạy

1. **Attach 2 datasets** vào notebook (Add Input → Search):
   - Competition raw data (có `train.parquet`, `validation.parquet`, `test.parquet`)
   - Feature cache dataset đã upload (có 7 file: features + style + v5.pkl + codebert probas)
2. **Sửa 2 biến `COMP_DIR` và `CACHE_DIR`** ở cell "Setup paths" bên dưới.
3. **Run All**.

Output submissions sẽ nằm ở `/kaggle/working/` — Kaggle tự snapshot khi notebook kết thúc.

## 1. Clone repo + install dependencies

In [ ]:
!git clone https://github.com/ThinhSama234/SemEval.git
%cd SemEval
!pip install -q lightgbm catboost xgboost

## 2. Setup paths

**Sửa 2 biến bên dưới** cho đúng với tên dataset bạn attach.

In [ ]:
import os

# =============================================================================
# EDIT THESE TWO PATHS BEFORE RUNNING
# =============================================================================
# Path tới dataset raw của competition (chứa train.parquet, validation.parquet, test.parquet)
COMP_DIR  = '/kaggle/input/semeval-2026-task-13'            # ← sửa
# Path tới dataset cache features đã upload
CACHE_DIR = '/kaggle/input/semeval-features-cache'          # ← sửa

# Output directory (Kaggle mặc định)
OUT_DIR   = '/kaggle/working'

# =============================================================================
# Paths (không cần sửa)
# =============================================================================
PATHS = {
    # raw competition data
    'train_data':  f'{COMP_DIR}/train.parquet',
    'val_data':    f'{COMP_DIR}/validation.parquet',
    'test_data':   f'{COMP_DIR}/test.parquet',
    # cached feature matrices
    'train_feat':  f'{CACHE_DIR}/train_features_ml_ready.parquet',
    'val_feat':    f'{CACHE_DIR}/val_features_ml_ready.parquet',
    'test_feat':   f'{CACHE_DIR}/test_features_ml_ready.parquet',
    # style features for v5
    'val_style':   f'{CACHE_DIR}/val_style_features.parquet',
    'test_style':  f'{CACHE_DIR}/test_style_features.parquet',
    # pre-trained artifacts
    'v5_pkl':      f'{CACHE_DIR}/taskA_hybrid_v5.pkl',
    'cb_test':     f'{CACHE_DIR}/test_proba.npy',
    # outputs
    'v10_pkl':     f'{OUT_DIR}/taskA_v10.pkl',
    'v10_sub':     f'{OUT_DIR}/submission_v10.csv',
    'v10_proba':   f'{OUT_DIR}/submission_v10_proba.npy',
}

# Verify every input file exists
print('Verifying inputs...')
missing = []
for key, path in PATHS.items():
    if key in ('v10_pkl', 'v10_sub', 'v10_proba'):  # outputs, don't check yet
        continue
    ok = os.path.exists(path)
    print(f'  [{"OK" if ok else "MISSING"}] {key}: {path}')
    if not ok:
        missing.append(key)

if missing:
    raise FileNotFoundError(
        f'\n{len(missing)} input files missing: {missing}\n'
        f'Check COMP_DIR and CACHE_DIR above, and make sure both datasets are attached.'
    )
print('\nAll inputs present ✓')

## 3. Phase 1 — Train v10 (language-robust gradient ensemble)

Fork của v7 nhưng bỏ 8 features (7 language-shifted + perplexity). Feature count 75.

**Expected**: val F1 ~0.96-0.98, CPU ~20-40 phút.

Script tự sinh 3 submission ngay sau train:
- `submission_v10.csv` — val-tuned threshold
- `submission_v10_q50.csv` — quantile 50% AI (safety)
- `submission_v10_q55.csv` — quantile 55% AI (matches train ratio)

In [ ]:
!python train_v10_lang_robust.py \
  --train_feat {PATHS['train_feat']} \
  --val_feat   {PATHS['val_feat']} \
  --test_feat  {PATHS['test_feat']} \
  --train_data {PATHS['train_data']} \
  --val_data   {PATHS['val_data']} \
  --test_data  {PATHS['test_data']} \
  --model_out  {PATHS['v10_pkl']} \
  --submission_out {PATHS['v10_sub']}

## 4. Phase 2 — Ensemble pipeline

Chạy voting + majority + stacking dùng v5 + v10 + CodeBERT. Thời gian ~5 phút.

Sinh thêm các submission sau:

**Soft voting (rank-avg):**
- `submission_vote_v10cb_q50.csv` / `_q55.csv` — bỏ v5
- `submission_vote_v5v10cb_q50.csv` / `_q55.csv` — cả 3 models
- `submission_vote_w244_q50.csv` / `_q55.csv` — 0.2·v5 + 0.4·v10 + 0.4·cb

**Majority vote (label):**
- `submission_maj_v5v10cb.csv`
- `submission_and_v10cb.csv`

**Stacking (LR meta-learner):**
- `submission_stack_v5v10_q50.csv` / `_q55.csv`

In [ ]:
!python ensemble_pipeline.py \
  --v10_pkl    {PATHS['v10_pkl']} \
  --v10_test   {PATHS['v10_proba']} \
  --v5_pkl     {PATHS['v5_pkl']} \
  --cb_test    {PATHS['cb_test']} \
  --test_style {PATHS['test_style']} \
  --val_style  {PATHS['val_style']} \
  --val_feat   {PATHS['val_feat']} \
  --val_data   {PATHS['val_data']} \
  --out_dir    {OUT_DIR}

## 5. Tổng hợp submissions + priority

In [ ]:
import glob, pandas as pd

# List all submission CSVs produced
subs = sorted(glob.glob(f'{OUT_DIR}/submission_*.csv'))
print(f'Generated {len(subs)} submission CSVs:\n')

rows = []
for p in subs:
    try:
        df = pd.read_csv(p)
        ai_pct = (df['label'] == 1).mean() * 100
        rows.append({'file': os.path.basename(p), 'n': len(df), 'AI%': f'{ai_pct:.2f}'})
    except Exception as e:
        rows.append({'file': os.path.basename(p), 'n': '?', 'AI%': f'ERR {e}'})

summary = pd.DataFrame(rows)
print(summary.to_string(index=False))

print('\n' + '=' * 60)
print('Priority submit order to Kaggle (theo thứ tự thử):')
print('=' * 60)
print('''
  1. submission_vote_v5v10cb_q55.csv     3-way rank-avg, 55% AI  [safest]
  2. submission_vote_v10cb_q50.csv       v10+CodeBERT (drop v5)
  3. submission_stack_v5v10_q50.csv      learned meta-weights
  4. submission_maj_v5v10cb.csv          naive majority vote
  5. submission_v10_q50.csv              v10 alone (control)
  6. submission_v10.csv                  v10 val-tuned threshold
''')

print(f'\nAll files are in {OUT_DIR} — Kaggle auto-snapshots them when this notebook finishes.')

## 6. (Optional) Inspect v10 test proba distribution

Kiểm tra xem v10 có calibrated tốt không. Nếu mean proba > 0.85 → model vẫn over-confident → ưu tiên submit bản `_q50` thay vì `submission_v10.csv`.

In [ ]:
import numpy as np

p_v10 = np.load(PATHS['v10_proba'])
p_cb  = np.load(PATHS['cb_test'])

print(f'v10 test proba stats:')
print(f'  mean   = {p_v10.mean():.4f}')
print(f'  median = {np.median(p_v10):.4f}')
print(f'  std    = {p_v10.std():.4f}')
print()
print('  AI% at various thresholds:')
for t in [0.3, 0.5, 0.7, 0.85, 0.93, 0.95]:
    print(f'    t={t}: {(p_v10 >= t).mean() * 100:.2f}% AI')

print(f'\nCodeBERT test proba for reference:')
print(f'  mean={p_cb.mean():.4f} median={np.median(p_cb):.4f}')
print()
print('  Goal: v10 median should be closer to 0.5 than v7 was (v7 had median=0.9994).')
print('  If v10 median > 0.85 → still over-confident → prefer _q50 submission.')

## 7. Reference — previous LB scores (từ log submissions cũ)

| Submission | LB F1 | Approach |
|---|---:|---|
| v5 (IF+CNB hybrid) | **0.535** | simple, balanced predictions |
| CodeBERT t=0.93 | 0.451 | neural + threshold calibration |
| v6 ensemble | 0.372 | 4-model soft-vote on style features |
| v9 multi-IF | 0.295 | 5 IF groups on feature subsets |
| v7 (full 76-feat ensemble) | 0.254 | XGB+LGB+CatBoost — *overfit Python/tab features* |

**Hypothesis for v10**: kỳ vọng **0.40-0.50+** (beats v7, có thể sát v5).

**Hypothesis for ensembles (vote/stack)**: có thể beat v5's 0.535 nếu v10 đạt ≥ 0.40.